# CNN-RecKAN — Epochs=20, Seed=2

In [ ]:
import math
import random
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# ============================================================
# 1. Reproducibility and device
# ============================================================
SEED = 2

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ============================================================
# 2. Experiment configuration
# ============================================================
BATCH_SIZE = 128
EPOCHS = 20
LR = 1e-3
WEIGHT_DECAY = 1e-4
GRAD_CLIP = 1.0

FEATURE_DIM = 128
NUM_CLASSES = 10

RECKAN_ORDER = 5
CNN_HIDDEN_DIM = 55

RECURRENCE_LR_MULTIPLIER = 0.25
COEFF_BOUND = 2.5

# ============================================================
# 3. Fashion-MNIST data
# ============================================================
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.2860,), (0.3530,))
])

train_dataset = datasets.FashionMNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.FashionMNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=256,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

# ============================================================
# 4. Identical CNN backbone for both models
# ============================================================
class CNNBackbone(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.Conv2d(32, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout(0.10),

            nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.Conv2d(64, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout(0.15),

            nn.Conv2d(64, FEATURE_DIM, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(FEATURE_DIM),
            nn.ReLU(inplace=True),

            nn.AdaptiveAvgPool2d((1, 1))
        )

    def forward(self, x):
        return self.features(x).flatten(1)

# ============================================================
# 5. Standard CNN classifier
#
# Parameter count:
# 128*55 + 55 + 55*10 + 10 = 7,655
# ============================================================
class MLPClassifier(nn.Module):
    def __init__(self, in_features=FEATURE_DIM,
                 hidden_features=CNN_HIDDEN_DIM,
                 num_classes=NUM_CLASSES):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(in_features, hidden_features),
            nn.ReLU(inplace=True),
            nn.Dropout(0.20),
            nn.Linear(hidden_features, num_classes)
        )

    def forward(self, x):
        return self.net(x)

class StandardCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = CNNBackbone()
        self.classifier = MLPClassifier()

    def forward(self, x):
        return self.classifier(self.backbone(x))

# ============================================================
# 6. RecKAN classifier
#
# R_0(x) = 0
# R_1(x) = 1
# R_{n+1}(x) =
#     (a*x^2 + b*x + c)R_n(x) + (d*x + e)R_{n-1}(x)
#
# Parameter count:
# weight: 128*(5+1)*10 = 7,680
# bias: 10
# recurrence coefficients: 5
# total: 7,695
# ============================================================
class RecKANClassifier(nn.Module):
    def __init__(self, in_features=FEATURE_DIM,
                 num_classes=NUM_CLASSES,
                 order=RECKAN_ORDER,
                 coeff_bound=COEFF_BOUND):
        super().__init__()

        self.in_features = in_features
        self.num_classes = num_classes
        self.order = order
        self.coeff_bound = coeff_bound

        def inverse_tanh(value):
            normalized = max(min(value / coeff_bound, 0.999), -0.999)
            return 0.5 * math.log((1.0 + normalized) / (1.0 - normalized))

        # Start at the exact Chebyshev-U recurrence:
        # R_{n+1}=2x R_n - R_{n-1}, R_0=0, R_1=1.
        self.raw_a = nn.Parameter(torch.tensor(inverse_tanh(0.0)))
        self.raw_b = nn.Parameter(torch.tensor(inverse_tanh(2.0)))
        self.raw_c = nn.Parameter(torch.tensor(inverse_tanh(0.0)))
        self.raw_d = nn.Parameter(torch.tensor(inverse_tanh(0.0)))
        self.raw_e = nn.Parameter(torch.tensor(inverse_tanh(-1.0)))

        self.weight = nn.Parameter(
            torch.empty(in_features, order + 1, num_classes)
        )
        self.bias = nn.Parameter(torch.zeros(num_classes))

        nn.init.xavier_uniform_(self.weight)

    def recurrence_coefficients(self):
        a = self.coeff_bound * torch.tanh(self.raw_a)
        b = self.coeff_bound * torch.tanh(self.raw_b)
        c = self.coeff_bound * torch.tanh(self.raw_c)
        d = self.coeff_bound * torch.tanh(self.raw_d)
        e = self.coeff_bound * torch.tanh(self.raw_e)
        return a, b, c, d, e

    def forward(self, features):
        x = torch.tanh(features)

        a, b, c, d, e = self.recurrence_coefficients()

        r0 = torch.zeros_like(x)
        r1 = torch.ones_like(x)
        basis = [r0, r1]

        for _ in range(1, self.order):
            r_next = (
                (a * x.square() + b * x + c) * basis[-1]
                + (d * x + e) * basis[-2]
            )

            scale = r_next.detach().abs().amax().clamp_min(1e-6)
            r_next = r_next / scale
            basis.append(r_next)

        basis = torch.stack(basis, dim=-1)

        return torch.einsum("bik,ikc->bc", basis, self.weight) + self.bias

class CNNRecKAN(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = CNNBackbone()
        self.classifier = RecKANClassifier()

    def forward(self, x):
        return self.classifier(self.backbone(x))

# ============================================================
# 7. Utilities
# ============================================================
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def count_classifier_parameters(model):
    return sum(
        p.numel() for p in model.classifier.parameters()
        if p.requires_grad
    )

@torch.no_grad()
def evaluate(model, loader):
    model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        logits = model(images)
        loss = F.cross_entropy(logits, labels)

        total_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, 100.0 * correct / total

def train_one_epoch(model, loader, optimizer):
    model.train()

    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        logits = model(images)
        loss = F.cross_entropy(logits, labels)

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, 100.0 * correct / total

def make_optimizer(model, use_reckan=False):
    if not use_reckan:
        return torch.optim.AdamW(
            model.parameters(),
            lr=LR,
            weight_decay=WEIGHT_DECAY
        )

    recurrence_parameters = [
        model.classifier.raw_a,
        model.classifier.raw_b,
        model.classifier.raw_c,
        model.classifier.raw_d,
        model.classifier.raw_e
    ]

    recurrence_ids = {id(p) for p in recurrence_parameters}

    regular_parameters = [
        p for p in model.parameters()
        if id(p) not in recurrence_ids
    ]

    return torch.optim.AdamW(
        [
            {"params": regular_parameters, "lr": LR},
            {
                "params": recurrence_parameters,
                "lr": LR * RECURRENCE_LR_MULTIPLIER
            }
        ],
        weight_decay=WEIGHT_DECAY
    )

def train_model(model, name, use_reckan=False):
    model = model.to(device)
    optimizer = make_optimizer(model, use_reckan=use_reckan)

    best_accuracy = -1.0
    best_epoch = 0
    best_state = None
    history = []

    print("\n" + "=" * 82)
    print(name)
    print(f"Backbone parameters:   {count_parameters(model.backbone):,}")
    print(f"Classifier parameters: {count_classifier_parameters(model):,}")
    print(f"Total parameters:      {count_parameters(model):,}")
    print("=" * 82)

    for epoch in range(1, EPOCHS + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer)
        test_loss, test_acc = evaluate(model, test_loader)

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "train_accuracy": train_acc,
            "test_loss": test_loss,
            "test_accuracy": test_acc
        })

        if test_acc > best_accuracy:
            best_accuracy = test_acc
            best_epoch = epoch
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }

        print(
            f"Epoch {epoch:02d}/{EPOCHS} | "
            f"Train: {train_acc:6.2f}% | "
            f"Test: {test_acc:6.2f}% | "
            f"Test loss: {test_loss:.4f}"
        )

    model.load_state_dict(best_state)

    result = {
        "name": name,
        "model": model,
        "history": history,
        "backbone_parameters": count_parameters(model.backbone),
        "classifier_parameters": count_classifier_parameters(model),
        "total_parameters": count_parameters(model),
        "best_test_accuracy": best_accuracy,
        "best_epoch": best_epoch
    }

    if use_reckan:
        a, b, c, d, e = model.classifier.recurrence_coefficients()

        result["learned_recurrence"] = {
            "a": a.item(),
            "b": b.item(),
            "c": c.item(),
            "d": d.item(),
            "e": e.item()
        }

    return result

# ============================================================
# 8. Construct models and verify parameter matching
# ============================================================
torch.manual_seed(SEED)
standard_cnn = StandardCNN()

torch.manual_seed(SEED)
cnn_reckan = CNNRecKAN()

standard_classifier_params = count_classifier_parameters(standard_cnn)
reckan_classifier_params = count_classifier_parameters(cnn_reckan)

relative_gap = (
    abs(standard_classifier_params - reckan_classifier_params)
    / reckan_classifier_params
) * 100.0

print("\nParameter matching check")
print(f"Standard CNN classifier: {standard_classifier_params:,}")
print(f"CNN-RecKAN classifier:   {reckan_classifier_params:,}")
print(f"Classifier difference:    {abs(standard_classifier_params - reckan_classifier_params):,}")
print(f"Relative difference:      {relative_gap:.3f}%")

# ============================================================
# 9. Train both models
# ============================================================
torch.manual_seed(SEED)
standard_result = train_model(
    standard_cnn,
    name="Standard CNN (matched MLP classifier)",
    use_reckan=False
)

torch.manual_seed(SEED)
reckan_result = train_model(
    cnn_reckan,
    name="CNN-RecKAN (matched RecKAN classifier)",
    use_reckan=True
)

# ============================================================
# 10. Report results
# ============================================================
print("\n" + "=" * 82)
print("FINAL RESULTS: FASHION-MNIST")
print("=" * 82)

print(
    f"{'Model':<42}"
    f"{'Total params':>15}"
    f"{'Best test acc.':>18}"
    f"{'Epoch':>8}"
)
print("-" * 82)

for result in [standard_result, reckan_result]:
    print(
        f"{result['name']:<42}"
        f"{result['total_parameters']:>15,}"
        f"{result['best_test_accuracy']:>17.2f}%"
        f"{result['best_epoch']:>8}"
    )

accuracy_difference = (
    reckan_result["best_test_accuracy"]
    - standard_result["best_test_accuracy"]
)

print("-" * 82)
print(f"CNN-RecKAN minus Standard CNN: {accuracy_difference:+.2f} percentage points")

print("\nLearned RecKAN recurrence coefficients:")
for key, value in reckan_result["learned_recurrence"].items():
    print(f"  {key} = {value:+.4f}")